<a href="https://colab.research.google.com/github/hadiah115-tech/code-switching-codesaviours-si26-hadia/blob/main/SI26_Week7_hadia.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import torch

print("GPU available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU")

GPU available: True
GPU: Tesla T4


In [3]:
import pandas as pd

df = pd.read_csv("dataset.csv")

print(df.head())
print("\nDataset shape:", df.shape)
print("\nColumns:", df.columns.tolist())
print("\nLabels:")
print(df["label"].value_counts())

                    sentence   word label
0  Aaj ka din bohot busy tha    Aaj   URD
1  Aaj ka din bohot busy tha     ka   URD
2  Aaj ka din bohot busy tha    din   URD
3  Aaj ka din bohot busy tha  bohot   URD
4  Aaj ka din bohot busy tha   busy   ENG

Dataset shape: (1358, 3)

Columns: ['sentence', 'word', 'label']

Labels:
label
ENG    868
URD    490
Name: count, dtype: int64


In [4]:
!pip install -q transformers datasets seqeval accelerate scikit-learn

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 2.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done


In [5]:
import pandas as pd
import torch

from sklearn.model_selection import train_test_split

label2id = {
    "URD": 0,
    "ENG": 1,
    "MIX": 2
}

id2label = {
    0: "URD",
    1: "ENG",
    2: "MIX"
}

print(label2id)
print(id2label)

{'URD': 0, 'ENG': 1, 'MIX': 2}
{0: 'URD', 1: 'ENG', 2: 'MIX'}


In [13]:
df = df.dropna(
    subset=["sentence", "word", "label"]
).copy()

df["sentence"] = df["sentence"].astype(str)
df["word"] = df["word"].astype(str)
df["label"] = df["label"].astype(str)

df = df[
    df["label"].isin(["URD", "ENG", "MIX"])
].copy()

print("Dataset after cleaning:")
print("Rows:", len(df))

print("\nLabels:")
print(df["label"].value_counts())

print("\nMissing values:")
print(df[["sentence", "word", "label"]].isna().sum())

Dataset after cleaning:
Rows: 1357

Labels:
label
ENG    867
URD    490
Name: count, dtype: int64

Missing values:
sentence    0
word        0
label       0
dtype: int64


In [14]:
label2id = {
    "URD": 0,
    "ENG": 1,
    "MIX": 2
}

id2label = {
    0: "URD",
    1: "ENG",
    2: "MIX"
}

print("label2id:", label2id)
print("id2label:", id2label)

label2id: {'URD': 0, 'ENG': 1, 'MIX': 2}
id2label: {0: 'URD', 1: 'ENG', 2: 'MIX'}


In [15]:
sentences = df.groupby("sentence").apply(
    lambda x: {
        "words": x["word"].tolist(),
        "labels": x["label"].tolist()
    }
).tolist()

print("Total sentences:", len(sentences))

print("\nFirst sentence:")
print(sentences[0])

Total sentences: 150

First sentence:
{'words': ['Aaj', 'hum', 'new', 'strategy', 'discuss', 'karein', 'ge', 'team', 'alignment', 'crucial'], 'labels': ['URD', 'URD', 'ENG', 'ENG', 'ENG', 'URD', 'URD', 'ENG', 'ENG', 'ENG']}


/tmp/ipykernel_899/440321133.py:1: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  sentences = df.groupby("sentence").apply(


In [16]:
from sklearn.model_selection import train_test_split

train_data, test_data = train_test_split(
    sentences,
    test_size=0.2,
    random_state=42
)

print("Training sentences:", len(train_data))
print("Testing sentences:", len(test_data))

Training sentences: 120
Testing sentences: 30


In [17]:
from transformers import (
    AutoTokenizer,
    AutoModelForTokenClassification
)

model_name = "xlm-roberta-base"

tokenizer = AutoTokenizer.from_pretrained(
    model_name
)

model = AutoModelForTokenClassification.from_pretrained(
    model_name,
    num_labels=3,
    id2label=id2label,
    label2id=label2id
)

print("Tokenizer and model loaded successfully!")

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] XLMRobertaForTokenClassification LOAD REPORT from: xlm-roberta-base
Key                         | Status     | 
----------------------------+------------+-
lm_head.layer_norm.bias     | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
lm_head.dense.weight        | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
classifier.weight           | MISSING    | 
classifier.bias             | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Tokenizer and model loaded successfully!


In [18]:
from datasets import Dataset

def to_hf_dataset(data):
    return Dataset.from_dict({
        "words": [d["words"] for d in data],
        "labels": [d["labels"] for d in data]
    })

train_ds = to_hf_dataset(train_data)
test_ds = to_hf_dataset(test_data)

print("Train dataset:")
print(train_ds)

print("\nTest dataset:")
print(test_ds)

Train dataset:
Dataset({
    features: ['words', 'labels'],
    num_rows: 120
})

Test dataset:
Dataset({
    features: ['words', 'labels'],
    num_rows: 30
})


In [19]:
def tokenize_and_align_labels(examples):

    tokenized = tokenizer(
        examples["words"],
        truncation=True,
        is_split_into_words=True
    )

    labels = []

    for i, label in enumerate(examples["labels"]):

        word_ids = tokenized.word_ids(batch_index=i)

        label_ids = []
        previous_word_id = None

        for word_id in word_ids:

            if word_id is None:
                label_ids.append(-100)

            elif word_id != previous_word_id:
                label_ids.append(
                    label2id[label[word_id]]
                )

            else:
                label_ids.append(-100)

            previous_word_id = word_id

        labels.append(label_ids)

    tokenized["labels"] = labels

    return tokenized

In [23]:
train_ds = train_ds.map(
    tokenize_and_align_labels,
    batched=True
)

test_ds = test_ds.map(
    tokenize_and_align_labels,
    batched=True
)

print("Tokenization completed!")
print(train_ds)

Map:   0%|          | 0/120 [00:00<?, ? examples/s]

Map:   0%|          | 0/30 [00:00<?, ? examples/s]

Tokenization completed!
Dataset({
    features: ['words', 'labels', 'input_ids', 'attention_mask'],
    num_rows: 120
})


In [22]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_steps=10,
    load_best_model_at_end=True,
    report_to="none"
)

print("Training arguments ready!")

Training arguments ready!


In [24]:
from transformers import DataCollatorForTokenClassification

data_collator = DataCollatorForTokenClassification(
    tokenizer=tokenizer
)

print("Data collator ready!")

Data collator ready!


In [26]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=test_ds,
    processing_class=tokenizer,
    data_collator=data_collator
)

print("Trainer ready!")

Trainer ready!


In [27]:
print("Starting training...")

trainer.train()

print("Training complete!")

Starting training...


Epoch,Training Loss,Validation Loss
1,0.695844,0.041924
2,0.071518,0.051292
3,0.049882,0.020520
4,0.029223,0.028654
5,0.015264,0.028003


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Training complete!


In [28]:
results = trainer.evaluate()

print("Evaluation Results:")
print(results)

Training Loss,Validation Loss,Epoch
0.015264,0.020520,5


Evaluation Results:
{'eval_loss': 0.020520051941275597}


In [30]:
from sklearn.metrics import classification_report, f1_score
import numpy as np

# Get predictions
predictions = trainer.predict(test_ds)

preds = predictions.predictions
labels = predictions.label_ids

# Get predicted class IDs
pred_ids = np.argmax(preds, axis=2)

# Store valid predictions and labels
all_predictions = []
all_labels = []

for prediction, label in zip(pred_ids, labels):

    for pred, lab in zip(prediction, label):

        # Ignore special tokens (-100)
        if lab != -100:
            all_predictions.append(pred)
            all_labels.append(lab)

# Convert IDs to label names
all_predictions = [
    id2label[pred] for pred in all_predictions
]

all_labels = [
    id2label[lab] for lab in all_labels
]

# Proper token-level classification report
print(
    classification_report(
        all_labels,
        all_predictions,
        labels=["URD", "ENG", "MIX"],
        zero_division=0
    )
)

# Individual F1 scores
f1_scores = f1_score(
    all_labels,
    all_predictions,
    labels=["URD", "ENG", "MIX"],
    average=None,
    zero_division=0
)

print("\nIndividual F1 Scores:")
print(f"URD F1: {f1_scores[0]:.4f}")
print(f"ENG F1: {f1_scores[1]:.4f}")
print(f"MIX F1: {f1_scores[2]:.4f}")

              precision    recall  f1-score   support

         URD       1.00      0.99      1.00       102
         ENG       0.99      1.00      1.00       169
         MIX       0.00      0.00      0.00         0

    accuracy                           1.00       271
   macro avg       0.66      0.66      0.66       271
weighted avg       1.00      1.00      1.00       271


Individual F1 Scores:
URD F1: 0.9951
ENG F1: 0.9971
MIX F1: 0.0000


In [31]:
print("All dataset labels:")
print(df["label"].value_counts())

print("\nTraining labels:")
train_labels = [label for item in train_data for label in item["labels"]]
print(pd.Series(train_labels).value_counts())

print("\nTesting labels:")
test_labels = [label for item in test_data for label in item["labels"]]
print(pd.Series(test_labels).value_counts())

All dataset labels:
label
ENG    867
URD    490
Name: count, dtype: int64

Training labels:
ENG    698
URD    388
Name: count, dtype: int64

Testing labels:
ENG    169
URD    102
Name: count, dtype: int64


In [32]:
print("Total sentences:", df["sentence"].nunique())

print("\nSample sentences:")
for sentence in df["sentence"].drop_duplicates().head(20):
    print(sentence)

Total sentences: 150

Sample sentences:
Aaj ka din bohot busy tha
Yaar seriously I can not even right now
Bhai kal mera presentation hai still not prepared
Khana kha liya I was literally waiting for you
Aaj mera mood nahi hai for anything
Main raste mein hoon please wait for ten minutes
Is weather mein aik hot cup of chai is mandatory
Assignment submit kar di hai or still working on it
Kal ki meeting ka time reschedule ho gaya hai
Bohot traffic tha so I missed the first lecture
Mujhe yeh movie dekhni hai looks very interesting
Please pass me that water bottle bohot pyas lagi hai
Itna tension mat lo everything will be fine
Laptop hang ho gaya hai I need to restart it
Yeh wala design looks aesthetic and modern
Suno message check karo I sent you the file
It is so hot today bilkul bahar nikalne ka dil nahi kar raha
Is code mein bug hai can you help me debug it
Chai break ka time ho gaya hai let us go outside
Main kal shopping par ja rahi hoon with my sister


In [33]:
print(df.groupby("sentence")["label"].nunique().value_counts())

label
2    150
Name: count, dtype: int64


In [34]:
model.save_pretrained("./code-switching-model")
tokenizer.save_pretrained("./code-switching-model")

print("Model saved successfully!")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Model saved successfully!


In [39]:
from huggingface_hub import login

login()

In [40]:
from huggingface_hub import whoami

print(whoami())

{'type': 'user', 'id': '6a4619bf15e14ca75b3b1aef', 'name': 'hadia-tech', 'fullname': 'Hadia Malik', 'canPay': False, 'billingMode': 'prepaid', 'periodEnd': 1788220800, 'isPro': False, 'avatarUrl': '/avatars/f3de3efe8c68c72d7aec7db035bd5c78.svg', 'orgs': [], 'auth': {'type': 'oauth', 'expiresAt': '2026-09-12T10:05:13.000Z'}}


In [41]:
repo_name = "code-switching-codesaviours-si26-hadia"

model.push_to_hub(repo_name)
tokenizer.push_to_hub(repo_name)

print("Model uploaded successfully!")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...d9onvpy/model.safetensors:   0%|          | 13.6kB / 1.11GB            

README.md:   0%|          | 0.00/5.17k [00:00<?, ?B/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...mppbuf54n9/tokenizer.json: 100%|##########| 17.1MB / 17.1MB            

Model uploaded successfully!


In [43]:
print(
    "https://huggingface.co/hadia-tech/"
    + repo_name
)

https://huggingface.co/hadia-tech/code-switching-codesaviours-si26-hadia
